# Notebook 04 - Evaluation

**Ramdas Chaugale - 40494109**  
**ECS8056 Themed Research Project**  
Five models are run over the 15 held-out images at two input
sizes, each with and without post-processing, giving **twenty configurations**.

Everything here is per-image and computed on predictions, unlike the training-loop averages
reviewed in notebook 03. Dice, IoU, precision, recall, connectivity, error split, probability
calibration, lesion size, per-animal behaviour and the RCC8D spatial relation are all reported.

The 256 and native halves are written out separately so either can be read on its own.

Notebooks run in order: 01 → 02 → 03 → 04 → 05.

## 1. Setup

Paths resolve from the project root. Each run is a dictionary carrying its input size, closing
kernel and checkpoint folder, so the two halves of the notebook differ only in which one they read.

In [ ]:
import os
import sys
import pickle
import time

import numpy as np
import matplotlib.pyplot as plt
import cv2
import torch
from PIL import Image

# this notebook runs from notebooks/, so the project root is one level up
PROJECT_ROOT = os.path.dirname(os.path.abspath(os.getcwd()))

DATA_DIR        = os.path.join(PROJECT_ROOT, 'data')
ENDPOINT_FOLDER = os.path.join(DATA_DIR, 'endpoint')
ANNO_FOLDER     = os.path.join(DATA_DIR, 'annotated',
                               'endpoint_annotated')
MASK_FOLDER     = os.path.join(DATA_DIR, 'masks')

OCTF_CODE      = os.path.join(PROJECT_ROOT, 'reconstruction',
                              'OCT-F_Reconstruction', 'code')
FINETUNE_CKPT  = os.path.join(PROJECT_ROOT, 'reconstruction',
                              'fine_tuned_reconstruction.pth')
RESIDUAL_CACHE = os.path.join(PROJECT_ROOT, 'reconstruction', 'residual_cache.pkl')

IMAGES_DIR = os.path.join(PROJECT_ROOT, 'outputs', 'images')
os.makedirs(IMAGES_DIR, exist_ok=True)

print("Project root:", PROJECT_ROOT)

In [ ]:
# the architectures live in architectures.py - the same module the training
# run imported on the HPC. it is in code file
for candidate in [os.path.join(PROJECT_ROOT, 'code'), os.path.join(PROJECT_ROOT, 'src')]:
    if os.path.exists(os.path.join(candidate, 'architectures.py')):
        sys.path.insert(0, candidate)
        break
sys.path.insert(0, OCTF_CODE)

SEED = 40494109
np.random.seed(SEED)
torch.manual_seed(SEED)
device = 'cuda' if torch.cuda.is_available() else 'cpu'

plt.rcParams.update({
    'figure.dpi': 100, 'savefig.dpi': 150, 'font.size': 11,
    'axes.titlesize': 12, 'axes.titleweight': 'bold', 'axes.labelsize': 11,
    'axes.spines.top': False, 'axes.spines.right': False,
    'axes.grid': True, 'grid.alpha': 0.25, 'legend.frameon': False,
    'figure.titlesize': 14, 'figure.titleweight': 'bold',
})

# prediction red, truth green, correct yellow used in every overlay below
PRED_RGB  = [255, 80, 80]
TRUTH_RGB = [46, 200, 90]
SMOOTH    = 1.0

print(f"Seed: {SEED}")
print(f"Device: {device}")

In [ ]:
# one descriptor per run, carrying everything that differs between them
RUN_256 = {'label': '256x256',
           'suffix': '_256',
           'size': (256, 256),
           'pp_kernel': 15,
           'ckpt_dir': os.path.join(PROJECT_ROOT, 'reconstruction', 'checkpoints_256')}

RUN_NATIVE = {'label': '496x768',
              'suffix': '',
              'size': (496, 768),
              'pp_kernel': 29,
              'ckpt_dir': os.path.join(PROJECT_ROOT, 'reconstruction', 'checkpoints')}

for run in [RUN_256, RUN_NATIVE]:
    assert os.path.exists(run['ckpt_dir']), f"MISSING: {run['ckpt_dir']}"
    print(f"{run['label']:<10} closing kernel {run['pp_kernel']:<3} {run['ckpt_dir']}")

Two run configurations, differing only in input geometry and the size-scaled closing kernel: 15 at 256 and 29 at native. All other settings are shared, so resolution is the only variable between
them.

## 2. Validation split

The 15 held-out images, reproduced from the seed rather than stored, and asserted disjoint from the
59 training images.

In [ ]:
# the same seed and the same sorted starting order reproduce the training split,
# so these 15 images are genuinely the held-out set
anno_files = sorted([f for f in os.listdir(ANNO_FOLDER) if f.endswith('.tif')])

np.random.seed(SEED)
shuffled = anno_files.copy()
np.random.shuffle(shuffled)
train_files = shuffled[:59]
val_files = shuffled[59:]

assert len(val_files) == 15
assert set(train_files).isdisjoint(val_files)

print(f"Train: {len(train_files)}   Validation: {len(val_files)}")

In [ ]:
print("The 15 held-out images")
for filename in val_files:
    print(f"  {filename[:-4]}")

15 validation images, disjoint from the 59 training images by assertion. They span eight animals,
but most animals contribute only one image, which is what makes the per-animal table in Sections 9
and 15 fragile.

## 3. Model instantiation

The fine-tuned foundation model, the cached residuals, and the ten segmentation checkpoints.
Loading is strict, so a mismatch between a class definition and its saved weights raises rather
than silently loading a different model.

In [ ]:
from architectures import (SimpleUNet, HybridUNet, OCTFSegmentation,
                        ModifiedOCTFSegmentation)
from architectureV2Res_new import ArchitectureV2

# models 3 and 4 copy their encoder blocks out of the fine-tuned OCT-F model,
# so that has to be rebuilt first
ft_model = ArchitectureV2().to(device)
ft_checkpoint = torch.load(FINETUNE_CKPT, map_location=device, weights_only=False)
ft_model.load_state_dict(ft_checkpoint['model_state_dict'])
ft_model.eval()

with open(RESIDUAL_CACHE, 'rb') as f:
    residual_cache = pickle.load(f)

print(f"Fine-tuned OCT-F loaded from epoch {ft_checkpoint['epoch']}")
print(f"Residuals loaded: {len(residual_cache)}")

In [ ]:
# file name and input type for each model, the same in both runs
MODEL_INFO = {
    'Plain U-Net':         ('plain_unet.pth', 'single'),
    'Hybrid U-Net':        ('hybrid_unet.pth', 'hybrid'),
    'OCT-F Backbone':      ('octf_backbone.pth', 'single'),
    'Modified OCT-F 2ch':  ('modified_octf_2ch.pth', 'hybrid'),
    'Soft Erosion Hybrid': ('hybrid_soft_erosion.pth', 'hybrid'),
}

models_256 = {}
models_native = {}
for run, store in [(RUN_256, models_256), (RUN_NATIVE, models_native)]:
    for name in MODEL_INFO:
        if name == 'Plain U-Net':
            model = SimpleUNet()
        elif name == 'OCT-F Backbone':
            model = OCTFSegmentation(ft_model)
        elif name == 'Modified OCT-F 2ch':
            model = ModifiedOCTFSegmentation(ft_model)
        else:
            model = HybridUNet()
        checkpoint = torch.load(os.path.join(run['ckpt_dir'], MODEL_INFO[name][0]),
                                map_location=device, weights_only=False)
        # strict by default, so a mismatch between definition and weights raises
        model.load_state_dict(checkpoint['model_state_dict'])
        model.eval()
        store[name] = {'model': model.to(device), 'checkpoint': checkpoint}

print(f"Loaded {len(models_256)} models at 256 and {len(models_native)} at native")

All ten load under strict matching, which establishes that the class definitions in
`architectures.py` are the ones that were trained. Residuals come from the cache written by `gpu_01`,
never recomputed.

## 4. Metrics and evaluation

The loaders, the post-processing, the per-image metrics and the inference loop. These are long and
used by every section below at both input sizes, so they are written once here, immediately above
the first evaluation they serve.

IoU is derived per image as `D / (2 − D)` and only then averaged; averaging Dice first and
converting afterwards gives a different number.

In [ ]:
# the evaluation machinery. it is long and every section below uses it at both
# input sizes, so it is written once here, immediately above the first evaluation it serves
def load_gray(filename, size):
    """The endpoint scan at one run's size, as float 0 to 1."""
    height, width = size
    gray = np.array(Image.open(os.path.join(ENDPOINT_FOLDER, filename)))[:, :, 0]
    return cv2.resize(gray, (width, height)).astype(np.float32) / 255.0


def load_mask(filename, size):
    """The ground-truth mask at one run's size, as float 0 or 1."""
    height, width = size
    mask = cv2.imread(os.path.join(MASK_FOLDER, filename.replace('.tif', '_mask.png')),
                      cv2.IMREAD_GRAYSCALE)
    resized = cv2.resize(mask, (width, height), interpolation=cv2.INTER_NEAREST)
    return (resized > 127).astype(np.float32)


def load_residual(filename, size):
    """The cached residual at one run's size."""
    height, width = size
    return cv2.resize(residual_cache[filename], (width, height)).astype(np.float32)


def post_process(binary, kernel_size):
    """Closing to bridge nearby fragments, then keep only the largest blob."""
    kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (kernel_size, kernel_size))
    closed = cv2.morphologyEx(binary.astype(np.uint8), cv2.MORPH_CLOSE, kernel)
    count, labels = cv2.connectedComponents(closed)
    if count <= 1:
        return closed.astype(np.float32)

    biggest = 1
    biggest_size = 0
    for label in range(1, count):
        size = (labels == label).sum()
        if size > biggest_size:
            biggest_size = size
            biggest = label
    return (labels == biggest).astype(np.float32)


def scores(prediction, truth):
    """Every per-image number the tables below report.

    Dice is the harmonic mean of precision and recall, so it can stay flat while
    those two move in opposite directions. Both are therefore kept separately.
    IoU is derived per image as D / (2 - D) and only averaged afterwards.
    """
    p = prediction.ravel()
    t = truth.ravel()
    intersection = (p * t).sum()

    dice = (2 * intersection + SMOOTH) / (p.sum() + t.sum() + SMOOTH)
    precision = (intersection + SMOOTH) / (p.sum() + SMOOTH)
    recall = (intersection + SMOOTH) / (t.sum() + SMOOTH)

    # one blob is expected: a laser lesion is a single region
    components = max(0, cv2.connectedComponents(prediction.astype(np.uint8))[0] - 1)

    return {'dice': float(dice),
            'iou': float(dice / (2 - dice)),
            'precision': float(precision),
            'recall': float(recall),
            'cc': int(components),
            'missed': float((1 - recall) * 100),
            'extra': float((p.sum() - intersection) / max(t.sum(), 1) * 100)}


def evaluate_all(models, run):
    """Every model of one run, giving ten configurations."""
    output = {}
    for name in models:
        input_type = MODEL_INFO[name][1]
        raw = {'dice': [], 'iou': [], 'precision': [], 'recall': [], 'cc': [],
               'missed': [], 'extra': [], 'gt_size': [], 'time': [],
               'ground_truth': [], 'prediction': [], 'probability': []}
        processed = {key: [] for key in raw}

        for filename in val_files:
            image = load_gray(filename, run['size'])
            truth = load_mask(filename, run['size'])
            if input_type == 'hybrid':
                stacked = np.stack([image, load_residual(filename, run['size'])])
            else:
                stacked = image[None]

            start_time = time.time()
            with torch.no_grad():
                probability = models[name]['model'](
                    torch.from_numpy(stacked)[None].to(device))
            elapsed = time.time() - start_time
            probability = probability[0, 0].cpu().numpy()

            hard = (probability > 0.5).astype(np.float32)
            cleaned = post_process(hard, run['pp_kernel'])
            size_percent = truth.sum() / truth.size * 100

            for store, mask_out in [(raw, hard), (processed, cleaned)]:
                for key, value in scores(mask_out, truth).items():
                    store[key].append(value)
                store['gt_size'].append(size_percent)
                store['time'].append(elapsed)
                store['ground_truth'].append(truth)
                store['prediction'].append(mask_out)
                store['probability'].append(probability)

        output[name] = raw
        output[name + ' + PP'] = processed
    return output


def count_fragmented(result):
    """How many of the 15 predictions broke into more than one blob."""
    total = 0
    for components in result['cc']:
        if components > 1:
            total += 1
    return total


def order_by_dice(results):
    """Configuration labels, highest mean Dice first."""
    means = {}
    for label in results:
        means[label] = np.mean(results[label]['dice'])
    ranked = []
    for label in results:
        ranked.append((means[label], label))
    ranked.sort(reverse=True)
    return [label for score, label in ranked]


start = time.time()
results_256 = evaluate_all(models_256, RUN_256)
print(f"Evaluated {len(results_256)} configurations at 256x256 "
      f"in {time.time() - start:.0f}s")

In [ ]:
start = time.time()
results_native = evaluate_all(models_native, RUN_NATIVE)
print(f"Evaluated {len(results_native)} configurations at native "
      f"in {time.time() - start:.0f}s")

In [ ]:
order_256 = order_by_dice(results_256)
order_native = order_by_dice(results_native)

print(f"Leading at 256:    {order_256[0]}")
print(f"Leading at native: {order_native[0]}")

Twenty configurations: five models x two input sizes x two post-processing states. The leading
configuration differs between the two runs, which Section 20 examines.

## 5. Results at 256 x 256

The main table, the spread behind it, and the distributions those means are computed from. The
spread matters more than the ordering.

In [ ]:
print(f"{'Configuration':<26} {'Dice':>6} {'Std':>6} {'IoU':>6} {'Prec':>6} "
      f"{'Recall':>7} {'CC>1':>6} {'Miss%':>6} {'Extra%':>7} {'ms':>6}")
print("=" * 92)
for label in order_256:
    r = results_256[label]
    print(f"{label:<26} {np.mean(r['dice']):>6.3f} {np.std(r['dice']):>6.3f} "
          f"{np.mean(r['iou']):>6.3f} {np.mean(r['precision']):>6.3f} "
          f"{np.mean(r['recall']):>7.3f} {count_fragmented(r):>4}/15 "
          f"{np.mean(r['missed']):>6.1f} {np.mean(r['extra']):>7.1f} "
          f"{np.mean(r['time']) * 1000:>6.0f}")

Soft Erosion Hybrid + PP is highest at **0.867**. The top five configurations span **0.016**,
which is inside the standard error of about 0.013 at n = 15 so the ordering of those five is not
meaningful.

In [ ]:
print(f"{'Configuration':<26} {'Dice':>7} {'Std':>7} {'Min':>7} {'Max':>7}")
print("=" * 58)
for label in order_256:
    dice = np.array(results_256[label]['dice'])
    print(f"{label:<26} {dice.mean():>7.3f} {dice.std():>7.3f} "
          f"{dice.min():>7.3f} {dice.max():>7.3f}")

Per-configuration minima span 0.41 to 0.73, so within-configuration variance across images is
larger than the between-configuration differences in the table above.

In [ ]:
# short names keep ten labels legible side by side on one axis
SHORT_NAME = {
    'Plain U-Net':         'Plain',
    'Hybrid U-Net':        'Hybrid',
    'OCT-F Backbone':      'OCT-F',
    'Modified OCT-F 2ch':  'Mod OCT-F',
    'Soft Erosion Hybrid': 'Soft Erosion',
}

dice_lists = []
iou_lists = []
labels = []
for label in order_256:
    dice_lists.append(results_256[label]['dice'])
    iou_lists.append(results_256[label]['iou'])
    # ten configurations share one axis, so the model names are shortened and
    # every label is given the same two-line shape. the previous form left the
    # raw variants on one line and the + PP variants on two, so adjacent
    # labels printed over each other
    if label.endswith(' + PP'):
        base = label[:-5]
        variant = '+ PP'
    else:
        base = label
        variant = 'raw'
    labels.append(SHORT_NAME[base] + '\n' + variant)

fig, axes = plt.subplots(1, 2, figsize=(17, 5))
for ax, data, name in [(axes[0], dice_lists, 'Dice'), (axes[1], iou_lists, 'IoU')]:
    box = ax.boxplot(data, tick_labels=labels, patch_artist=True, widths=0.55)
    for patch in box['boxes']:
        patch.set_facecolor('#2F6FA8')
        patch.set_alpha(0.55)
    for median in box['medians']:
        median.set_color('#C0392B')
        median.set_linewidth(1.8)
    ax.set_ylabel(name)
    ax.set_title(f"{name} across 15 validation images")
    ax.tick_params(axis='x', labelsize=8)
    ax.grid(alpha=0.25, axis='y')

fig.suptitle(f"Per-image segmentation quality at {RUN_256['label']}, "
             f"the distributions overlap heavily", y=1.02)
fig.tight_layout()
fig.savefig(os.path.join(IMAGES_DIR, '04_boxplot_256.png'), bbox_inches='tight')
plt.show()

**How to read it.** Each box summarises the **15 validation images** for one configuration, not a
single number. The box covers the middle half of those images, the red line is the median, the
whiskers reach the furthest image that is still typical, and each circle is one image that sits
outside that range.

The point of the figure is that the boxes overlap almost completely, and that **any one box is
taller than the gaps between boxes** the spread from image to image is larger than the difference
between models. The low circles are the same hard image recurring in every configuration.

Interquartile ranges overlap across configurations. The graphical form makes explicit what the
table understates: these configurations are not separable on 15 images.

## 6. Post-processing at 256 x 256

What morphological closing followed by largest-component selection changes, model by model, and
what it looks like on an image that needed it.

In [ ]:
print(f"{'Model':<22} {'CC>1 raw':>9} {'CC>1 PP':>8} {'Dice raw':>9} "
      f"{'Dice PP':>8} {'Change':>8}")
print("=" * 68)
for name in MODEL_INFO:
    raw = results_256[name]
    processed = results_256[name + ' + PP']
    raw_dice = np.mean(raw['dice'])
    pp_dice = np.mean(processed['dice'])
    print(f"{name:<22} {count_fragmented(raw):>7}/15 "
          f"{count_fragmented(processed):>6}/15 "
          f"{raw_dice:>9.3f} {pp_dice:>8.3f} {pp_dice - raw_dice:>+8.3f}")

Post-processing yields **0/15** multi-component predictions for all five models, with Dice change
between −0.002 and +0.005. The connectivity constraint is essentially free.

In [ ]:
# the first image where the leading model fragmented, before and after cleaning
name = order_256[0].replace(' + PP', '')
raw = results_256[name]
processed = results_256[name + ' + PP']

index = 0
for i in range(len(val_files)):
    if raw['cc'][i] > 1:
        index = i
        break

gray = load_gray(val_files[index], RUN_256['size'])
panels = [(gray, 'gray', 'Original scan'),
          (raw['ground_truth'][index], 'gray', 'Ground truth'),
          (raw['prediction'][index], 'gray',
           f"Raw: {raw['cc'][index]} components, Dice {raw['dice'][index]:.3f}"),
          (processed['prediction'][index], 'gray',
           f"After PP: {processed['cc'][index]} component, "
           f"Dice {processed['dice'][index]:.3f}")]

fig, axes = plt.subplots(1, 4, figsize=(19, 3.4))
for ax, (picture, cmap, title) in zip(axes, panels):
    ax.imshow(picture, cmap=cmap)
    ax.set_title(title, fontsize=10)
    ax.axis('off')

fig.suptitle(f"Post-processing on {name} at {RUN_256['label']}, "
             f"{val_files[index][:-4]}", y=1.05)
fig.tight_layout()
fig.savefig(os.path.join(IMAGES_DIR, '04_postprocessing_effect_256.png'),
            bbox_inches='tight')
plt.show()

Closing followed by largest-component selection removes the spurious component while leaving the
main region intact, which is why the Dice cost is negligible.

## 7. Predictions at 256 x 256

The same image segmented by all five models, then the best, median and worst cases of the leading
configuration.

In [ ]:
# the middle validation image segmented by every model
index = len(val_files) // 2
gray = (load_gray(val_files[index], RUN_256['size']) * 255).astype(np.uint8)
ground_truth = results_256['Plain U-Net']['ground_truth'][index]

fig, axes = plt.subplots(2, 3, figsize=(17, 6.2))
axes = axes.ravel()

truth_overlay = cv2.cvtColor(gray, cv2.COLOR_GRAY2RGB)
truth_overlay[ground_truth > 0] = TRUTH_RGB
axes[0].imshow(truth_overlay)
axes[0].set_title(f"Ground truth\n{val_files[index][:-4]}", fontsize=10)

position = 1
for name in MODEL_INFO:
    prediction = results_256[name]['prediction'][index]
    overlay = cv2.cvtColor(gray, cv2.COLOR_GRAY2RGB)
    overlay[prediction > 0] = PRED_RGB
    axes[position].imshow(overlay)
    axes[position].contour(ground_truth, levels=[0.5], colors='#39FF6A',
                           linewidths=1.2)
    axes[position].set_title(f"{name}\nDice {results_256[name]['dice'][index]:.3f}, "
                             f"{results_256[name]['cc'][index]} component(s)", fontsize=10)
    position += 1

for ax in axes:
    ax.axis('off')

fig.suptitle(f"Red = prediction, green outline = ground truth, at {RUN_256['label']}",
             y=1.02)
fig.tight_layout()
fig.savefig(os.path.join(IMAGES_DIR, '04_all_models_comparison_256.png'),
            bbox_inches='tight')
plt.show()

In [ ]:
# best, median and worst by rank on the leading configuration
ranked = np.argsort(results_256[order_256[0]]['dice'])
cases_256 = {'best': int(ranked[-1]),
      'median': int(ranked[len(ranked) // 2]),
      'worst': int(ranked[0])}

leading = results_256[order_256[0]]
print(f"Leading configuration: {order_256[0]}")
print()
for case in cases_256:
    i = cases_256[case]
    print(f"{case:<8} {val_files[i][:-4]:<22} Dice {leading['dice'][i]:.3f}  "
          f"IoU {leading['iou'][i]:.3f}  lesion {leading['gt_size'][i]:.2f}% of frame")

Best and worst images are **1.17%** and **1.21%** of frame respectively effectively equal in
size, so difficulty is not explained by how large the lesion is.

In [ ]:
# one figure per case, drawn the same way
leading = results_256[order_256[0]]
for case in cases_256:
    i = cases_256[case]
    gray = (load_gray(val_files[i], RUN_256['size']) * 255).astype(np.uint8)
    ground_truth = leading['ground_truth'][i]

    overlay = cv2.cvtColor(gray, cv2.COLOR_GRAY2RGB)
    overlay[leading['prediction'][i] > 0] = PRED_RGB

    fig, axes = plt.subplots(1, 3, figsize=(16, 3.4))
    axes[0].imshow(gray, cmap='gray')
    axes[0].set_title('Original scan', fontsize=10)
    axes[1].imshow(ground_truth, cmap='gray')
    axes[1].set_title('Ground truth', fontsize=10)
    axes[2].imshow(overlay)
    axes[2].contour(ground_truth, levels=[0.5], colors='#39FF6A', linewidths=1.2)
    axes[2].set_title('Prediction (red) over truth (green)', fontsize=10)

    for ax in axes:
        ax.axis('off')

    fig.suptitle(f"{case.capitalize()} case at {RUN_256['label']}: {val_files[i][:-4]}, "
                 f"Dice {leading['dice'][i]:.3f}", y=1.06)
    fig.tight_layout()
    fig.savefig(os.path.join(IMAGES_DIR, '04_prediction_' + case + '_256.png'),
                bbox_inches='tight')
    plt.show()

In all three cases the lesion is localised correctly and the error is in boundary placement rather
than in finding the lesion at all.

## 8. Missed and extra at 256 x 256

The error split. Dice is the harmonic mean of precision and recall, so it can stay flat while those
two move in opposite directions this section separates them.

In [ ]:
leading = results_256[order_256[0]]
fig, axes = plt.subplots(1, 3, figsize=(17, 3.4))

position = 0
for case in cases_256:
    i = cases_256[case]
    gray = (load_gray(val_files[i], RUN_256['size']) * 255).astype(np.uint8)
    ground_truth = leading['ground_truth'][i] > 0
    prediction = leading['prediction'][i] > 0

    overlay = cv2.cvtColor(gray, cv2.COLOR_GRAY2RGB)
    overlay[ground_truth & prediction] = [255, 220, 0]
    overlay[ground_truth & ~prediction] = TRUTH_RGB
    overlay[~ground_truth & prediction] = [230, 40, 40]

    axes[position].imshow(overlay)
    axes[position].set_title(f"{case}: {val_files[i][:-4]}\n"
                             f"missed {leading['missed'][i]:.1f}%, "
                             f"extra {leading['extra'][i]:.1f}%", fontsize=10)
    axes[position].axis('off')
    position += 1

fig.suptitle(f"Yellow = correct, green = missed, red = extra, at {RUN_256['label']}",
             y=1.06)
fig.tight_layout()
fig.savefig(os.path.join(IMAGES_DIR, '04_under_over_segmentation_256.png'),
            bbox_inches='tight')
plt.show()

**How to read it.** Three colours, and only three things can happen to a pixel. **Yellow** is where
the prediction and the expert agree. **Green** is lesion the expert marked and the model missed.
**Red** is area the model added that the expert did not mark.

So yellow is correct, green is under-segmentation and red is over-segmentation. A perfect prediction
would be entirely yellow. The percentages in each title are the green and red areas as a share of
the true lesion.

In [ ]:
print(f"{'Configuration':<26} {'Missed %':>9} {'Extra %':>9} "
      f"{'Precision':>10} {'Recall':>8}")
print("=" * 66)
for label in order_256:
    print(f"{label:<26} {np.mean(results_256[label]['missed']):>9.1f} "
          f"{np.mean(results_256[label]['extra']):>9.1f} "
          f"{np.mean(results_256[label]['precision']):>10.3f} "
          f"{np.mean(results_256[label]['recall']):>8.3f}")

Plain U-Net produces the most excess area at **18.6%**, Hybrid U-Net the least at **13.0%**. The
two-channel models trade recall for precision: they miss more lesion but include less non-lesion tissue.

## 9. Probability, size and animal at 256 x 256

Three checks on what drives the numbers: how confident the model is, whether lesion size explains
performance, and whether any animal behaves differently from the rest.

In [ ]:
leading = results_256[order_256[0]]
i = cases_256['best']
gray = load_gray(val_files[i], RUN_256['size'])
probability = leading['probability'][i]

fig, axes = plt.subplots(1, 4, figsize=(19, 3.4))
axes[0].imshow(gray, cmap='gray')
axes[0].set_title('Original scan', fontsize=10)
heat = axes[1].imshow(probability, cmap='hot', vmin=0, vmax=1)
axes[1].set_title('Predicted probability', fontsize=10)
fig.colorbar(heat, ax=axes[1], fraction=0.03)
axes[2].imshow(probability > 0.5, cmap='gray')
axes[2].set_title('Thresholded at 0.5', fontsize=10)
axes[3].imshow(leading['ground_truth'][i], cmap='gray')
axes[3].set_title('Ground truth', fontsize=10)

for ax in axes:
    ax.axis('off')

fig.suptitle(f"Probability map at {RUN_256['label']}: {val_files[i][:-4]} "
             f"(one image of one model)", y=1.05)
fig.tight_layout()
fig.savefig(os.path.join(IMAGES_DIR, '04_confidence_heatmap_256.png'),
            bbox_inches='tight')
plt.show()

**How to read it.** The model does not output a mask directly it outputs a **probability for every
pixel**, between 0 and 1. The second panel shows those probabilities on a heat scale: black is near
0, bright is near 1. The third panel is that same map after cutting at 0.5, which is the mask
actually scored.

What to notice is how little of the map is uncertain the transition from black to bright happens
over a very narrow band at the lesion edge. The model is decisive nearly everywhere.
**This is one image of one model and should not be generalised.**

In [ ]:
probability = results_256[order_256[0]]['probability'][cases_256['best']]
uncertain = ((probability > 0.3) & (probability < 0.7)).sum()
confident = ((probability < 0.1) | (probability > 0.9)).sum()
total = probability.size

print(f"Image: {val_files[cases_256['best']][:-4]}")
print(f"Uncertain pixels (0.3 to 0.7): {uncertain:,} ({uncertain / total * 100:.2f}%)")
print(f"Confident pixels (below 0.1 or above 0.9): {confident:,} "
      f"({confident / total * 100:.2f}%)")

On this image, **99.77%** of pixels lie below 0.1 or above 0.9 and **0.10%** fall between 0.3 and
0.7. The model is confident almost everywhere, with uncertainty confined to the boundary. **One
image of one model this does not generalise.**

In [ ]:
leading = results_256[order_256[0]]
sizes = np.array(leading['gt_size'])

fig, axes = plt.subplots(1, 2, figsize=(14, 4.2))
axes[0].scatter(sizes, leading['dice'], s=55, alpha=0.85, color='#2F6FA8',
                edgecolor='white')
for i in range(len(val_files)):
    axes[0].annotate(val_files[i].split()[0], (sizes[i], leading['dice'][i]),
                     fontsize=7, alpha=0.65, xytext=(4, 4),
                     textcoords='offset points')
axes[0].set_xlabel('lesion size (% of frame)')
axes[0].set_ylabel('Dice')
axes[0].set_title(order_256[0])

for name in MODEL_INFO:
    axes[1].scatter(sizes, results_256[name]['dice'], s=32, alpha=0.75, label=name)
axes[1].set_xlabel('lesion size (% of frame)')
axes[1].set_ylabel('Dice')
axes[1].set_title('All five models')
axes[1].legend(fontsize=8)

fig.suptitle(f"Lesion size against segmentation quality at {RUN_256['label']}, "
             f"no relationship is established at n = 15", y=1.02)
fig.tight_layout()
fig.savefig(os.path.join(IMAGES_DIR, '04_size_vs_dice_256.png'), bbox_inches='tight')
plt.show()

**How to read it.** One point per validation image. Lesion size runs left to right, segmentation
quality bottom to top. If small lesions were harder, the points would climb from lower-left to
upper-right in a clear line.

The left panel labels each point with its animal; the right panel overlays all five models, one
colour each. **The scatter is loose in both.** The formal test is in the table above: a relationship
would need a correlation of 0.514 to be significant at 15 images, and nothing here reaches it in a
way that survives scrutiny.

In [ ]:
sizes = np.array(results_256[order_256[0]]['gt_size'])
print(f"{'Configuration':<26} {'Correlation':>12}")
print("=" * 40)
for label in order_256:
    correlation = np.corrcoef(sizes, results_256[label]['dice'])[0, 1]
    print(f"{label:<26} {correlation:>12.3f}")

Correlations span **0.142 to 0.453** at this input size. The two-tailed 5% critical value of |r|
at n = 15 is **0.514**, so none of them clears it.

In [ ]:
# most animals contribute a single validation image, so these means are fragile
animals = []
for filename in val_files:
    animal = filename.split()[0]
    if animal not in animals:
        animals.append(animal)
animals = sorted(animals)

header = f"{'Animal':<10} {'n':>3}"
for name in MODEL_INFO:
    header += f" {name[:9]:>9}"
print(header)
print("=" * 66)

for animal in animals:
    indexes = []
    for i in range(len(val_files)):
        if val_files[i].startswith(animal):
            indexes.append(i)
    row = f"{animal:<10} {len(indexes):>3}"
    for name in MODEL_INFO:
        values = []
        for i in indexes:
            values.append(results_256[name]['dice'][i])
        row += f" {np.mean(values):>9.3f}"
    print(row)

In [ ]:
x = np.arange(len(animals))
width = 0.15

fig, ax = plt.subplots(figsize=(13, 4.4))
position = 0
for name in MODEL_INFO:
    means = []
    for animal in animals:
        values = []
        for i in range(len(val_files)):
            if val_files[i].startswith(animal):
                values.append(results_256[name]['dice'][i])
        means.append(np.mean(values))
    ax.bar(x + position * width, means, width, label=name, alpha=0.9,
           edgecolor='white')
    position += 1

ax.set_xticks(x + width * 2)
ax.set_xticklabels(animals)
ax.set_xlabel('animal')
ax.set_ylabel('mean Dice')
ax.set_title(f"Per-animal performance at {RUN_256['label']}, "
             f"one animal behaves unlike the rest")
ax.legend(fontsize=8, ncol=2)
ax.grid(alpha=0.25, axis='y')

fig.tight_layout()
fig.savefig(os.path.join(IMAGES_DIR, '04_per_animal_256.png'), bbox_inches='tight')
plt.show()

**How to read it.** Eight animals across the bottom, five coloured bars each one per model and
bar height is the mean Dice for that animal.

Read across the groups first: seven of the eight sit together in a narrow band. Then read the odd
one out. **Most animals contribute only one validation image**, so a bar here is often a single
image rather than an average over many, and it should be read that way.

One subject scores below every other subject for all five models, **0.492 to 0.730**. Since
most animals contribute a single validation image, this is one image rather than a property of the
animal.

## 10. RCC8D at 256 x 256

The spatial relation between each prediction and its ground truth, which says *how* a prediction is
wrong rather than by how much. Argument order follows Porter et al. (2024), ground truth first
reversing it inverts every proper-part relation.

In [ ]:
# the module published with Porter et al. (2024) is used when it is importable
try:
    from RCC8D.rcc8d import RCC8D as PorterRCC8D
    official_rcc8d = PorterRCC8D()
    HAVE_OFFICIAL = True
    print("Using the published RCC8D module (Porter et al., 2024)")
except Exception as error:
    official_rcc8d = None
    HAVE_OFFICIAL = False
    print(f"Published RCC8D module unavailable ({type(error).__name__})")
    print("Local implementation only. Install with: pip install RCC8D")

In [ ]:
# used by both runs and by the cross-check below, so defined once here,
# immediately above the first table it produces
KERNEL_8 = np.ones((3, 3), np.uint8)
RELATIONS = ['EQ', 'TPP', 'NTPP', 'TPPi', 'NTPPi', 'PO', 'EC', 'DC']


def boundary_of(region):
    """Outer boundary of a region: its dilation minus itself."""
    grown = cv2.dilate(region.astype(np.uint8), KERNEL_8).astype(bool)
    return grown & ~region


def compute_rcc8d(ground_truth, prediction):
    """RCC8D relation of the ground truth with respect to the prediction.

    Argument order follows Porter et al. (2024), ground truth first. TPP and NTPP
    therefore mean the prediction overestimates; TPPi and NTPPi mean it
    underestimates. Getting this order wrong inverts every proper-part relation.
    """
    truth = ground_truth.astype(bool)
    predicted = prediction.astype(bool)

    if not predicted.any() or not truth.any():
        return 'DC'

    overlap = np.logical_and(truth, predicted).any()
    truth_outside = np.logical_and(truth, ~predicted).any()
    predicted_outside = np.logical_and(predicted, ~truth).any()
    boundaries_touch = np.logical_and(boundary_of(truth), boundary_of(predicted)).any()

    if not overlap:
        grown = cv2.dilate(predicted.astype(np.uint8), KERNEL_8).astype(bool)
        if np.logical_and(grown, truth).any():
            return 'EC'
        return 'DC'

    if not truth_outside and not predicted_outside:
        return 'EQ'
    if not truth_outside:
        return 'TPP' if boundaries_touch else 'NTPP'
    if not predicted_outside:
        return 'TPPi' if boundaries_touch else 'NTPPi'
    return 'PO'


def official_relation(ground_truth, prediction):
    """Relation from the published module, or None if it is unavailable."""
    if not HAVE_OFFICIAL:
        return None
    try:
        truth = (ground_truth > 0).astype(np.uint8) * 255
        predicted = (prediction > 0).astype(np.uint8) * 255
        return official_rcc8d.compare_images(truth, predicted).get('relation_name')
    except Exception:
        return None


header = f"{'Configuration':<26}"
for relation in RELATIONS:
    header += f" {relation:>6}"
print(header)
print("=" * 82)

for label in order_256:
    counts = {}
    for relation in RELATIONS:
        counts[relation] = 0
    for i in range(len(val_files)):
        found = compute_rcc8d(results_256[label]['ground_truth'][i],
                              results_256[label]['prediction'][i])
        counts[found] += 1
    row = f"{label:<26}"
    for relation in RELATIONS:
        row += f" {counts[relation]:>6}"
    print(row)

Nine of ten configurations return **PO** partial overlap on all 15 images; the exception
returns a single TPPi. Every prediction straddles the lesion boundary, so the error is boundary
placement rather than mislocalisation.

## 11. Results at native 496 x 768

The same tables at the acquisition geometry, written out again so this half of the notebook can be
read on its own.

In [ ]:
print(f"{'Configuration':<26} {'Dice':>6} {'Std':>6} {'IoU':>6} {'Prec':>6} "
      f"{'Recall':>7} {'CC>1':>6} {'Miss%':>6} {'Extra%':>7} {'ms':>6}")
print("=" * 92)
for label in order_native:
    r = results_native[label]
    print(f"{label:<26} {np.mean(r['dice']):>6.3f} {np.std(r['dice']):>6.3f} "
          f"{np.mean(r['iou']):>6.3f} {np.mean(r['precision']):>6.3f} "
          f"{np.mean(r['recall']):>7.3f} {count_fragmented(r):>4}/15 "
          f"{np.mean(r['missed']):>6.1f} {np.mean(r['extra']):>7.1f} "
          f"{np.mean(r['time']) * 1000:>6.0f}")

Plain U-Net + PP is highest at native with **0.855** a different configuration from the 256 run,
which is itself evidence that the ordering is not stable.

In [ ]:
print(f"{'Configuration':<26} {'Dice':>7} {'Std':>7} {'Min':>7} {'Max':>7}")
print("=" * 58)
for label in order_native:
    dice = np.array(results_native[label]['dice'])
    print(f"{label:<26} {dice.mean():>7.3f} {dice.std():>7.3f} "
          f"{dice.min():>7.3f} {dice.max():>7.3f}")

Minima separate sharply at native: **0.756** for Plain U-Net against **0.016** for Modified OCT-F
2ch. A single image drives that difference, as Section 22 shows.

In [ ]:
# short names keep ten labels legible side by side on one axis
SHORT_NAME = {
    'Plain U-Net':         'Plain',
    'Hybrid U-Net':        'Hybrid',
    'OCT-F Backbone':      'OCT-F',
    'Modified OCT-F 2ch':  'Mod OCT-F',
    'Soft Erosion Hybrid': 'Soft Erosion',
}

dice_lists = []
iou_lists = []
labels = []
for label in order_native:
    dice_lists.append(results_native[label]['dice'])
    iou_lists.append(results_native[label]['iou'])
    # ten configurations share one axis, so the model names are shortened and
    # every label is given the same two-line shape. the previous form left the
    # raw variants on one line and the + PP variants on two, so adjacent
    # labels printed over each other
    if label.endswith(' + PP'):
        base = label[:-5]
        variant = '+ PP'
    else:
        base = label
        variant = 'raw'
    labels.append(SHORT_NAME[base] + '\n' + variant)

fig, axes = plt.subplots(1, 2, figsize=(17, 5))
for ax, data, name in [(axes[0], dice_lists, 'Dice'), (axes[1], iou_lists, 'IoU')]:
    box = ax.boxplot(data, tick_labels=labels, patch_artist=True, widths=0.55)
    for patch in box['boxes']:
        patch.set_facecolor('#2F6FA8')
        patch.set_alpha(0.55)
    for median in box['medians']:
        median.set_color('#C0392B')
        median.set_linewidth(1.8)
    ax.set_ylabel(name)
    ax.set_title(f"{name} across 15 validation images")
    ax.tick_params(axis='x', labelsize=8)
    ax.grid(alpha=0.25, axis='y')

fig.suptitle(f"Per-image segmentation quality at {RUN_NATIVE['label']}, "
             f"the distributions overlap heavily", y=1.02)
fig.tight_layout()
fig.savefig(os.path.join(IMAGES_DIR, '04_boxplot.png'), bbox_inches='tight')
plt.show()

**How to read it.** As at 256: one box per configuration, built from the 15 validation images. Box
is the middle half, red line the median, circles are individual images falling outside the usual
range.

**The detail worth pointing at is an absence.** Plain U-Net has no low outlier here, while every
other model drops a circle to between 0.55 and 0.01. That circle is the same image `scan_09` and Plain U-Net is the only model that handled it. That single image is why it leads at native.

Distributions are wider at native, with extended lower whiskers on several configurations, so the
means are less representative of typical performance than at 256.

## 12. Post-processing at native

The same post-processing measurement at native geometry, where fragmentation is more common.

In [ ]:
print(f"{'Model':<22} {'CC>1 raw':>9} {'CC>1 PP':>8} {'Dice raw':>9} "
      f"{'Dice PP':>8} {'Change':>8}")
print("=" * 68)
for name in MODEL_INFO:
    raw = results_native[name]
    processed = results_native[name + ' + PP']
    raw_dice = np.mean(raw['dice'])
    pp_dice = np.mean(processed['dice'])
    print(f"{name:<22} {count_fragmented(raw):>7}/15 "
          f"{count_fragmented(processed):>6}/15 "
          f"{raw_dice:>9.3f} {pp_dice:>8.3f} {pp_dice - raw_dice:>+8.3f}")

Post-processing yields **0/15** for all five models, with Dice gains of 0.001 to 0.005. Every gain
is positive here, unlike at 256.

In [ ]:
# the first image where the leading model fragmented, before and after cleaning
name = order_native[0].replace(' + PP', '')
raw = results_native[name]
processed = results_native[name + ' + PP']

index = 0
for i in range(len(val_files)):
    if raw['cc'][i] > 1:
        index = i
        break

gray = load_gray(val_files[index], RUN_NATIVE['size'])
panels = [(gray, 'gray', 'Original scan'),
          (raw['ground_truth'][index], 'gray', 'Ground truth'),
          (raw['prediction'][index], 'gray',
           f"Raw: {raw['cc'][index]} components, Dice {raw['dice'][index]:.3f}"),
          (processed['prediction'][index], 'gray',
           f"After PP: {processed['cc'][index]} component, "
           f"Dice {processed['dice'][index]:.3f}")]

fig, axes = plt.subplots(1, 4, figsize=(19, 3.4))
for ax, (picture, cmap, title) in zip(axes, panels):
    ax.imshow(picture, cmap=cmap)
    ax.set_title(title, fontsize=10)
    ax.axis('off')

fig.suptitle(f"Post-processing on {name} at {RUN_NATIVE['label']}, "
             f"{val_files[index][:-4]}", y=1.05)
fig.tight_layout()
fig.savefig(os.path.join(IMAGES_DIR, '04_postprocessing_effect.png'),
            bbox_inches='tight')
plt.show()

The same mechanism at the larger geometry, with a closing kernel of 29 rather than 15.

## 13. Predictions at native

The same figures at native. The leading configuration differs between runs, so the best, median and
worst cases are selected against a different model than at 256.

In [ ]:
# the middle validation image segmented by every model
index = len(val_files) // 2
gray = (load_gray(val_files[index], RUN_NATIVE['size']) * 255).astype(np.uint8)
ground_truth = results_native['Plain U-Net']['ground_truth'][index]

fig, axes = plt.subplots(2, 3, figsize=(17, 6.2))
axes = axes.ravel()

truth_overlay = cv2.cvtColor(gray, cv2.COLOR_GRAY2RGB)
truth_overlay[ground_truth > 0] = TRUTH_RGB
axes[0].imshow(truth_overlay)
axes[0].set_title(f"Ground truth\n{val_files[index][:-4]}", fontsize=10)

position = 1
for name in MODEL_INFO:
    prediction = results_native[name]['prediction'][index]
    overlay = cv2.cvtColor(gray, cv2.COLOR_GRAY2RGB)
    overlay[prediction > 0] = PRED_RGB
    axes[position].imshow(overlay)
    axes[position].contour(ground_truth, levels=[0.5], colors='#39FF6A',
                           linewidths=1.2)
    axes[position].set_title(f"{name}\nDice {results_native[name]['dice'][index]:.3f}, "
                             f"{results_native[name]['cc'][index]} component(s)", fontsize=10)
    position += 1

for ax in axes:
    ax.axis('off')

fig.suptitle(f"Red = prediction, green outline = ground truth, at {RUN_NATIVE['label']}",
             y=1.02)
fig.tight_layout()
fig.savefig(os.path.join(IMAGES_DIR, '04_all_models_comparison.png'),
            bbox_inches='tight')
plt.show()

In [ ]:
# best, median and worst by rank on the leading configuration
ranked = np.argsort(results_native[order_native[0]]['dice'])
cases_native = {'best': int(ranked[-1]),
      'median': int(ranked[len(ranked) // 2]),
      'worst': int(ranked[0])}

leading = results_native[order_native[0]]
print(f"Leading configuration: {order_native[0]}")
print()
for case in cases_native:
    i = cases_native[case]
    print(f"{case:<8} {val_files[i][:-4]:<22} Dice {leading['dice'][i]:.3f}  "
          f"IoU {leading['iou'][i]:.3f}  lesion {leading['gt_size'][i]:.2f}% of frame")

The leading configuration differs between runs, so these three images are not the same three as
at 256 they are selected against Plain U-Net + PP rather than Soft Erosion Hybrid + PP.

In [ ]:
# one figure per case, drawn the same way
leading = results_native[order_native[0]]
for case in cases_native:
    i = cases_native[case]
    gray = (load_gray(val_files[i], RUN_NATIVE['size']) * 255).astype(np.uint8)
    ground_truth = leading['ground_truth'][i]

    overlay = cv2.cvtColor(gray, cv2.COLOR_GRAY2RGB)
    overlay[leading['prediction'][i] > 0] = PRED_RGB

    fig, axes = plt.subplots(1, 3, figsize=(16, 3.4))
    axes[0].imshow(gray, cmap='gray')
    axes[0].set_title('Original scan', fontsize=10)
    axes[1].imshow(ground_truth, cmap='gray')
    axes[1].set_title('Ground truth', fontsize=10)
    axes[2].imshow(overlay)
    axes[2].contour(ground_truth, levels=[0.5], colors='#39FF6A', linewidths=1.2)
    axes[2].set_title('Prediction (red) over truth (green)', fontsize=10)

    for ax in axes:
        ax.axis('off')

    fig.suptitle(f"{case.capitalize()} case at {RUN_NATIVE['label']}: {val_files[i][:-4]}, "
                 f"Dice {leading['dice'][i]:.3f}", y=1.06)
    fig.tight_layout()
    fig.savefig(os.path.join(IMAGES_DIR, '04_prediction_' + case + '.png'),
                bbox_inches='tight')
    plt.show()

As at 256, the lesion is localised correctly in all three cases and the error is at the boundary.

## 14. Missed and extra at native

The error split at native geometry, where the precision-recall trade of the two-channel models is
most visible.

In [ ]:
leading = results_native[order_native[0]]
fig, axes = plt.subplots(1, 3, figsize=(17, 3.4))

position = 0
for case in cases_native:
    i = cases_native[case]
    gray = (load_gray(val_files[i], RUN_NATIVE['size']) * 255).astype(np.uint8)
    ground_truth = leading['ground_truth'][i] > 0
    prediction = leading['prediction'][i] > 0

    overlay = cv2.cvtColor(gray, cv2.COLOR_GRAY2RGB)
    overlay[ground_truth & prediction] = [255, 220, 0]
    overlay[ground_truth & ~prediction] = TRUTH_RGB
    overlay[~ground_truth & prediction] = [230, 40, 40]

    axes[position].imshow(overlay)
    axes[position].set_title(f"{case}: {val_files[i][:-4]}\n"
                             f"missed {leading['missed'][i]:.1f}%, "
                             f"extra {leading['extra'][i]:.1f}%", fontsize=10)
    axes[position].axis('off')
    position += 1

fig.suptitle(f"Yellow = correct, green = missed, red = extra, at {RUN_NATIVE['label']}",
             y=1.06)
fig.tight_layout()
fig.savefig(os.path.join(IMAGES_DIR, '04_under_over_segmentation.png'),
            bbox_inches='tight')
plt.show()

**How to read it.** The same three-colour scheme: **yellow** agrees with the expert, **green** is
lesion that was missed, **red** is area added that the expert did not mark.

Read the balance rather than the total. A model with more green than red is cautious and leaves
lesion behind; more red than green means it over-reaches. This is where the two-channel models
differ from the one-channel ones, even though their Dice scores are close.

In [ ]:
print(f"{'Configuration':<26} {'Missed %':>9} {'Extra %':>9} "
      f"{'Precision':>10} {'Recall':>8}")
print("=" * 66)
for label in order_native:
    print(f"{label:<26} {np.mean(results_native[label]['missed']):>9.1f} "
          f"{np.mean(results_native[label]['extra']):>9.1f} "
          f"{np.mean(results_native[label]['precision']):>10.3f} "
          f"{np.mean(results_native[label]['recall']):>8.3f}")

The two-channel models exchange recall for precision: Hybrid U-Net reaches **0.894** precision
against 0.849 for Plain U-Net, while its recall falls to **0.820** against 0.875. Dice is nearly
unchanged the residual moves where the error is, not how much of it there is.

## 15. Probability, size and animal at native

The same three checks at native geometry.

In [ ]:
leading = results_native[order_native[0]]
i = cases_native['best']
gray = load_gray(val_files[i], RUN_NATIVE['size'])
probability = leading['probability'][i]

fig, axes = plt.subplots(1, 4, figsize=(19, 3.4))
axes[0].imshow(gray, cmap='gray')
axes[0].set_title('Original scan', fontsize=10)
heat = axes[1].imshow(probability, cmap='hot', vmin=0, vmax=1)
axes[1].set_title('Predicted probability', fontsize=10)
fig.colorbar(heat, ax=axes[1], fraction=0.03)
axes[2].imshow(probability > 0.5, cmap='gray')
axes[2].set_title('Thresholded at 0.5', fontsize=10)
axes[3].imshow(leading['ground_truth'][i], cmap='gray')
axes[3].set_title('Ground truth', fontsize=10)

for ax in axes:
    ax.axis('off')

fig.suptitle(f"Probability map at {RUN_NATIVE['label']}: {val_files[i][:-4]} "
             f"(one image of one model)", y=1.05)
fig.tight_layout()
fig.savefig(os.path.join(IMAGES_DIR, '04_confidence_heatmap.png'),
            bbox_inches='tight')
plt.show()

**How to read it.** Four panels: the scan, the model's per-pixel **probability** on a heat scale
(black near 0, bright near 1), that map thresholded at 0.5, and the expert's mask for comparison.

The useful comparison is the third panel against the fourth. Any difference between them is the
error the Dice score measures. **One image of one model do not generalise from it.**

In [ ]:
probability = results_native[order_native[0]]['probability'][cases_native['best']]
uncertain = ((probability > 0.3) & (probability < 0.7)).sum()
confident = ((probability < 0.1) | (probability > 0.9)).sum()
total = probability.size

print(f"Image: {val_files[cases_native['best']][:-4]}")
print(f"Uncertain pixels (0.3 to 0.7): {uncertain:,} ({uncertain / total * 100:.2f}%)")
print(f"Confident pixels (below 0.1 or above 0.9): {confident:,} "
      f"({confident / total * 100:.2f}%)")

**99.73%** confident and **0.10%** uncertain on this image, essentially the same as at 256.
Again, one image of one model.

In [ ]:
leading = results_native[order_native[0]]
sizes = np.array(leading['gt_size'])

fig, axes = plt.subplots(1, 2, figsize=(14, 4.2))
axes[0].scatter(sizes, leading['dice'], s=55, alpha=0.85, color='#2F6FA8',
                edgecolor='white')
for i in range(len(val_files)):
    axes[0].annotate(val_files[i].split()[0], (sizes[i], leading['dice'][i]),
                     fontsize=7, alpha=0.65, xytext=(4, 4),
                     textcoords='offset points')
axes[0].set_xlabel('lesion size (% of frame)')
axes[0].set_ylabel('Dice')
axes[0].set_title(order_native[0])

for name in MODEL_INFO:
    axes[1].scatter(sizes, results_native[name]['dice'], s=32, alpha=0.75, label=name)
axes[1].set_xlabel('lesion size (% of frame)')
axes[1].set_ylabel('Dice')
axes[1].set_title('All five models')
axes[1].legend(fontsize=8)

fig.suptitle(f"Lesion size against segmentation quality at {RUN_NATIVE['label']}, "
             f"no relationship is established at n = 15", y=1.02)
fig.tight_layout()
fig.savefig(os.path.join(IMAGES_DIR, '04_size_vs_dice.png'), bbox_inches='tight')
plt.show()

**How to read it.** Each point is one validation image: lesion size across, Dice up. A real
size effect would show as points rising steadily from left to right.

The left panel is the leading configuration with the animal labelled; the right panel is all five
models together. **The vertical spread at any given size is large** images of nearly the same size
score very differently which is the visual form of the claim that lesion size does not explain
performance at this sample size.

In [ ]:
sizes = np.array(results_native[order_native[0]]['gt_size'])
print(f"{'Configuration':<26} {'Correlation':>12}")
print("=" * 40)
for label in order_native:
    correlation = np.corrcoef(sizes, results_native[label]['dice'])[0, 1]
    print(f"{label:<26} {correlation:>12.3f}")

Correlations span **0.259 to 0.572** at this geometry. One configuration exceeds the n = 15
critical value of 0.514 native Plain U-Net at r = 0.572, p = 0.026. Testing ten configurations at
the 5% level produces about one such result **by chance alone**, and it falls to r = 0.452,
p = 0.105 when the single hardest image is removed. **No relationship can be established at
n = 15.**

In [ ]:
# most animals contribute a single validation image, so these means are fragile
animals = []
for filename in val_files:
    animal = filename.split()[0]
    if animal not in animals:
        animals.append(animal)
animals = sorted(animals)

header = f"{'Animal':<10} {'n':>3}"
for name in MODEL_INFO:
    header += f" {name[:9]:>9}"
print(header)
print("=" * 66)

for animal in animals:
    indexes = []
    for i in range(len(val_files)):
        if val_files[i].startswith(animal):
            indexes.append(i)
    row = f"{animal:<10} {len(indexes):>3}"
    for name in MODEL_INFO:
        values = []
        for i in indexes:
            values.append(results_native[name]['dice'][i])
        row += f" {np.mean(values):>9.3f}"
    print(row)

In [ ]:
x = np.arange(len(animals))
width = 0.15

fig, ax = plt.subplots(figsize=(13, 4.4))
position = 0
for name in MODEL_INFO:
    means = []
    for animal in animals:
        values = []
        for i in range(len(val_files)):
            if val_files[i].startswith(animal):
                values.append(results_native[name]['dice'][i])
        means.append(np.mean(values))
    ax.bar(x + position * width, means, width, label=name, alpha=0.9,
           edgecolor='white')
    position += 1

ax.set_xticks(x + width * 2)
ax.set_xticklabels(animals)
ax.set_xlabel('animal')
ax.set_ylabel('mean Dice')
ax.set_title(f"Per-animal performance at {RUN_NATIVE['label']}, "
             f"one animal behaves unlike the rest")
ax.legend(fontsize=8, ncol=2)
ax.grid(alpha=0.25, axis='y')

fig.tight_layout()
fig.savefig(os.path.join(IMAGES_DIR, '04_per_animal.png'), bbox_inches='tight')
plt.show()

**How to read it.** One group per animal, five bars per group, height is mean Dice for that model on
that animal. Same colour means the same model throughout.

Seven groups are level with each other. **One subject is the exception** and the bars there collapse in
different amounts for different models, from 0.802 down to 0.014. That single group is the source of
almost all the apparent difference between models, and **it is one image, not an average**.

That subject returns **0.802, 0.553, 0.296, 0.014 and 0.464** across the five models, against
0.73 to 0.92 for every other animal. With most animals contributing a single validation image, one
image determines the ranking.

## 16. RCC8D at native

The same spatial-relation table at native, then the local implementation checked image by image
against the module published with Porter et al. (2024).

In [ ]:
header = f"{'Configuration':<26}"
for relation in RELATIONS:
    header += f" {relation:>6}"
print(header)
print("=" * 82)

for label in order_native:
    counts = {}
    for relation in RELATIONS:
        counts[relation] = 0
    for i in range(len(val_files)):
        found = compute_rcc8d(results_native[label]['ground_truth'][i],
                              results_native[label]['prediction'][i])
        counts[found] += 1
    row = f"{label:<26}"
    for relation in RELATIONS:
        row += f" {counts[relation]:>6}"
    print(row)

In [ ]:
# the local implementation checked against the published module, image by image
leading = results_native[order_native[0]]
agree = 0

for i in range(len(val_files)):
    local = compute_rcc8d(leading['ground_truth'][i], leading['prediction'][i])
    published = official_relation(leading['ground_truth'][i], leading['prediction'][i])
    if published == local:
        agree += 1
    print(f"  {val_files[i][:-4]:<22} local {local:<6} module {published}")

print()
print(f"Agreement with the published module: {agree}/{len(val_files)} images")

PO on almost every image; the two Modified OCT-F 2ch rows return one NTPPi, a prediction wholly
inside the lesion. The local implementation agrees with the published module on **15/15** images,
which validates the reimplementation.

## 17. Side by side

Every configuration at both input sizes, with the spread and the fragmentation counts alongside, so
the resolution comparison can be read in one place.

In [ ]:
print(f"{'Configuration':<26} {'256':>8} {'Native':>8} {'Diff':>8} "
      f"{'256 std':>8} {'Nat std':>8} {'256 CC':>7} {'Nat CC':>7}")
print("=" * 86)

for name in MODEL_INFO:
    for suffix in ['', ' + PP']:
        label = name + suffix
        small = results_256[label]
        large = results_native[label]
        small_dice = np.mean(small['dice'])
        large_dice = np.mean(large['dice'])
        print(f"{label:<26} {small_dice:>8.3f} {large_dice:>8.3f} "
              f"{small_dice - large_dice:>+8.3f} "
              f"{np.std(small['dice']):>8.3f} {np.std(large['dice']):>8.3f} "
              f"{count_fragmented(small):>5}/15 {count_fragmented(large):>5}/15")

Eight of ten score higher at 256, Plain U-Net excepted. Standard deviations are consistently
larger at native, and so are the fragmentation counts.

## 18. Distributions at both input sizes

The post-processed distributions side by side, on a shared vertical scale.

In [ ]:
# the same five models with post-processing, ordered the same way in both panels
labels = []
dice_256 = []
dice_native = []
for name in MODEL_INFO:
    labels.append(name.replace(' ', '\n', 1))
    dice_256.append(results_256[name + ' + PP']['dice'])
    dice_native.append(results_native[name + ' + PP']['dice'])

fig, axes = plt.subplots(1, 2, figsize=(15, 5), sharey=True)
for ax, data, title in [(axes[0], dice_256, 'Dice at 256 x 256'),
                        (axes[1], dice_native, 'Dice at 496 x 768')]:
    box = ax.boxplot(data, tick_labels=labels, patch_artist=True, widths=0.55)
    for patch in box['boxes']:
        patch.set_facecolor('#2F6FA8')
        patch.set_alpha(0.55)
    for median in box['medians']:
        median.set_color('#C0392B')
        median.set_linewidth(1.8)
    ax.set_title(title + ', with post-processing')
    ax.tick_params(axis='x', labelsize=9)
    ax.grid(alpha=0.25, axis='y')
axes[0].set_ylabel('Dice')

fig.suptitle('The same five models at both input sizes', y=1.02)
fig.tight_layout()
fig.savefig(os.path.join(IMAGES_DIR, '04_resolution_boxplot.png'), bbox_inches='tight')
plt.show()

**How to read it.** The same five models drawn twice trained and evaluated at 256 on the left, at
native geometry on the right with a shared vertical scale so the two panels can be compared
directly by eye.

Compare each model against **itself** across the two panels rather than against its neighbours. The
boxes are tighter on the left for four of the five, and the long tails on the right are the hard
image pulling those models down.

Distributions are tighter at 256 for every model except Plain U-Net. The extended lower whiskers
at native belong to the same image in each case.

## 19. Image by image

Every validation image at both input sizes, for all five post-processed models. This is the table
the project's central methodological finding comes from.

In [ ]:
print(f"{'Image':<22}", end='')
for name in MODEL_INFO:
    print(f" {name[:8]:>17}", end='')
print()
print(f"{'':<22}" + "        256  native" * len(MODEL_INFO))
print("=" * 112)

for i in range(len(val_files)):
    row = f"{val_files[i][:-4]:<22}"
    for name in MODEL_INFO:
        small = results_256[name + ' + PP']['dice'][i]
        large = results_native[name + ' + PP']['dice'][i]
        row += f" {small:>10.3f} {large:>6.3f}"
    print(row)

On **scan_09** the four non-Plain models fall to 0.016–0.678 at native while Plain U-Net
retains 0.802. Removing that one image moves Plain U-Net from first to third. Every other image
sits between 0.73 and 0.94 for every model.

## 20. Resolution agreement

The same numbers as a scatter: one point per image per model, native against 256.

In [ ]:
# one point per image per model: native on the x axis, 256 on the y axis
fig, ax = plt.subplots(figsize=(7, 7))

for name in MODEL_INFO:
    x = results_native[name + ' + PP']['dice']
    y = results_256[name + ' + PP']['dice']
    ax.scatter(x, y, s=45, alpha=0.8, label=name, edgecolor='white')

ax.plot([0, 1], [0, 1], color='0.4', linewidth=1.0, linestyle='--',
        label='equal at both sizes')
ax.set_xlabel('Dice at 496 x 768')
ax.set_ylabel('Dice at 256 x 256')
ax.set_title('Every image, every model, with post-processing')
ax.legend(fontsize=8, loc='lower right')

fig.tight_layout()
fig.savefig(os.path.join(IMAGES_DIR, '04_resolution_scatter.png'), bbox_inches='tight')
plt.show()

**How to read it.** Every point is **one image evaluated by one model**, with its native score across
and its 256 score up. The dashed diagonal is where the two would be equal.

So a point **above** the line did better at 256, and **below** the line did better at native. The
cluster sits in the top-right corner, meaning most images score well at both sizes. The handful of
points far to the left are cases that collapsed at native but held up at 256 the same hard image
under different models.

Points cluster on the identity line in the upper right; the outliers to the left are the same
image across four models. Agreement between the two runs is good except where one image is hard.

## 21. Results serialisation

The final tables written to `outputs/` for the paper and the supporting material. Written out
twice, once per run, so each file is produced by the code immediately above it.

In [ ]:
lines = []
lines.append(f"# Results Summary ({RUN_256['label']})")
lines.append("")
lines.append(f"Evaluated on {len(val_files)} held-out images, seed {SEED}.")
lines.append("")
lines.append("| Model | Params | In | Dice | Std | IoU | Prec | Recall | CC>1 | "
             "Missed % | Extra % | Time |")
lines.append("|-------|--------|----|------|-----|-----|------|--------|------|"
             "----------|---------|------|")

for label in order_256:
    result = results_256[label]
    name = label.replace(' + PP', '')
    config = models_256[name]['checkpoint']['config']
    lines.append(
        f"| {label} | {config['params'] / 1e6:.2f}M | {config['input_channels']}ch | "
        f"{np.mean(result['dice']):.3f} | {np.std(result['dice']):.3f} | "
        f"{np.mean(result['iou']):.3f} | {np.mean(result['precision']):.3f} | "
        f"{np.mean(result['recall']):.3f} | {count_fragmented(result)}/{len(val_files)} | "
        f"{np.mean(result['missed']):.1f} | {np.mean(result['extra']):.1f} | "
        f"{np.mean(result['time']) * 1000:.0f}ms |")

path = os.path.join(PROJECT_ROOT, 'outputs', 'results_summary_256.md')
with open(path, 'w', encoding='utf-8') as f:
    f.write("\n".join(lines) + "\n")
print("\n".join(lines))
print()
print(f"Wrote {path}")

In [ ]:
lines = []
lines.append(f"# Results Summary ({RUN_NATIVE['label']})")
lines.append("")
lines.append(f"Evaluated on {len(val_files)} held-out images, seed {SEED}.")
lines.append("")
lines.append("| Model | Params | In | Dice | Std | IoU | Prec | Recall | CC>1 | "
             "Missed % | Extra % | Time |")
lines.append("|-------|--------|----|------|-----|-----|------|--------|------|"
             "----------|---------|------|")

for label in order_native:
    result = results_native[label]
    name = label.replace(' + PP', '')
    config = models_native[name]['checkpoint']['config']
    lines.append(
        f"| {label} | {config['params'] / 1e6:.2f}M | {config['input_channels']}ch | "
        f"{np.mean(result['dice']):.3f} | {np.std(result['dice']):.3f} | "
        f"{np.mean(result['iou']):.3f} | {np.mean(result['precision']):.3f} | "
        f"{np.mean(result['recall']):.3f} | {count_fragmented(result)}/{len(val_files)} | "
        f"{np.mean(result['missed']):.1f} | {np.mean(result['extra']):.1f} | "
        f"{np.mean(result['time']) * 1000:.0f}ms |")

path = os.path.join(PROJECT_ROOT, 'outputs', 'results_summary.md')
with open(path, 'w', encoding='utf-8') as f:
    f.write("\n".join(lines) + "\n")
print("\n".join(lines))
print()
print(f"Wrote {path}")

In [ ]:
print("Written:")
print("  outputs/results_summary_256.md")
print("  outputs/results_summary.md")
print()
print("Figures saved in outputs/images/:")
for name in sorted(os.listdir(IMAGES_DIR)):
    if name.startswith('04_'):
        print(f"  {name}")

---

## Summary of notebook 04

Twenty configurations: five models, two input sizes, each with and without post-processing,
evaluated per image on the 15 held-out scans.

**No model is claimed as best.** At 256 the leading configuration reaches **0.867** and the top
five span **0.016**; at native the leader reaches **0.855**. With n = 15 the standard error is
about **0.013**, so differences below roughly **0.026** are not distinguishable. The boxplots make
this visible: the interquartile ranges overlap across nearly every configuration.

**One image decides which model appears to win.** On `scan_09` at native, four of the five
models fall to between 0.016 and 0.553 while Plain U-Net holds at 0.802. Remove that single image
and Plain U-Net drops from first to third. This is the most important methodological finding in the
project, and it is why no ranking here is presented as a result.

**Post-processing works and the loss-based constraint does not.** Closing followed by
largest-component selection takes every configuration to **0/15** fragmented at both input sizes,
costing at most 0.005 Dice.

**The residual channel changes the error profile, not the overlap.** Precision rises and recall
falls while Dice stays flat, so the two-channel models trade missed lesion for reduced excess.

**Lesion size does not explain performance.** Correlations span 0.142 to 0.453 at 256 and 0.259 to
0.572 at native. Testing ten post-processed configurations at the 5% level, one clears the n = 15
critical value of 0.514 about what chance alone produces and it falls below the threshold when
the single hardest image is removed. **No relationship can be established at n = 15**, which is a
weaker and more defensible claim than asserting there is none.

**Every prediction straddles the lesion boundary.** RCC8D returns partial overlap on essentially
every image, so the residual error is boundary placement rather than mislocalisation. The local
implementation agrees with the published module of Porter et al. (2024) on **15/15** images.